# ThousandWorlds quickstart


Run this notebook from the `thousandworlds` project root, that way downloads load into `./dataset/` next to your checkout.


In [ ]:
from pathlib import Path
import pandas as pd
import thousandworlds as tw
from thousandworlds.models.train_mean import TrainMean



Download the dataset archive into the repository. (The checkout already has metadata; this fills in the large array files.)


In [ ]:
tw.download_dataset(".")


Load the `single-complete` benchmark subset.


In [ ]:
data_dir = Path("dataset")
bundle = tw.load("single-complete", data_dir=data_dir)
stats = tw.load_stats("single-complete", data_dir)
bundle.X_train.shape, bundle.Y_train.shape, bundle.Y_test.shape


Check which output fields are present in the test set.


In [ ]:
pd.DataFrame({
    "field": bundle.field_names,
    "observed_in_test": bundle.field_mask_test.mean(axis=0),
}).head()


Fit the train-mean baseline and predict the held-out simulations.


In [ ]:
model = TrainMean().fit(
    bundle.Y_train,
    bundle.raw_field_names,
    stats,
    X_train=bundle.X_train,
    field_mask=bundle.field_mask_train,
)
pred = model.predict(bundle.X_test)


Score the predictions and display RMSE by physical variable.


In [ ]:
scores = tw.score_predictions(pred, bundle)
pd.Series(scores["rmse"]["per_variable"], name="rmse")
